In [1]:
import sys 
sys.path.insert(0, "../../../Personal-AI")
sys.path.insert(0, "../../")

from neo4j_functions import Neo4jConnection
from src import ReaderMetrics

import json
import requests
from langchain_huggingface import HuggingFaceEmbeddings
import os
from typing import List, Dict, Tuple
from time import sleep
from tqdm import tqdm
import numpy as np
import gc
import torch
import chromadb
from scipy.spatial import distance

####

EVAL_DATADIR = '../../data/qa_eval'

####

LLM_URL = "https://6708-109-252-76-222.ngrok-free.app/llama" 

NEO4J_URL ="bolt://31.207.47.254:7687"
NEO4J_USER = "neo4j"
NEO4J_PWD = "password"
NEO4J_DBNAME = "testdb"

####

LOG_DIR = "./logs"
SAVE_STAGE1_LOGDIR = f"{LOG_DIR}/stage1"
SAVE_STAGE2_LOGDIR = f"{LOG_DIR}/stage2"
SAVE_STAGE3_LOGDIR = f"{LOG_DIR}/stage3"
SAVE_STAGE4_LOGDIR = f"{LOG_DIR}/stage4"
SAVE_SCORES_LOGDIR = f"{LOG_DIR}/final"

LOG_TMPDATA_DIRNAME = "tmp"
LOG_META_FILENMAE = "metadata.json"
LOG_SCORES_DIRNAME = 'scores'

##### Utils

In [2]:
def generate(prompt: str) -> str:
    flag = True
    while flag:    
        try:
            response = requests.post(LLM_URL, params = {"prompt": prompt})
            resp = response.json()["response"]
            flag = False
        except (requests.ConnectionError, requests.ReadTimeout) as e:
            print("Connection error: ", str(e))
            sleep(1)

    return resp

def check_create_dir(new_dir_path: str):
    if os.path.exists(new_dir_path):
        print("Директория существует")
        raise ValueError
    else:
        os.mkdir(new_dir_path)

def save_json(data: Dict[str, object], save_path: str):
    dump = json.dumps(data, ensure_ascii=False, indent=1)
    with open(save_path, 'w', encoding='utf-8') as fd:
        fd.write(dump)

def load_json(load_path: str) -> Dict[str,object]:
    with open(load_path, 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())
    return data

def round5(number: float) -> float:
    return round(number, 5)

#### Stage 1: extracting entities from questions

In [3]:
prompt_extract_entities_template = '''You are an expert system that can extract key entities from text. Key entities is a noun or an object like persone, device, company and etc. Extract such entities from the given text and present the results in the following format: <entitie1> | <entitie2> | ... | <entitieN>. Generate only entities and dont return some additional text. Examples of texts and extracted entities are listed below:
Text 1: Kayla has positive, negative or neutral opinion about video of Xiaomi 10Pro?
Entities 1: Kayla | opinion | video | Xiaomi 10Pro.
Text 2: Which device is better in battery life: Apple or k30u?
Entities 2: device | battery life | Apple | k30u.
Text 3: The majority of speakers have positive, neutral or negative sentiment about screen of Samsung?
Entities 3: speakers | sentiment | screen | Samsung.
Text 4: Which people have positive opinion about video of Xiaomi 10Pro on 25.11.2020?
Entities 4: people | opinion | video | Xiaomi 10Pro | 25.11.2020.

Text: {text}
Entities: '''

S1_META = {
    'SAVE_ENTITIES_VERSION': "v2",
    'EXTRACT_PROMPT_TEMPLATE': prompt_extract_entities_template,
    'ENTITIES_PER_QUESTION': {}
}

In [5]:
def postproc_entity_extraction(raw_text: str) -> List[str]:    
    return list(filter(lambda item: len(item) > 0, list(map(lambda item: item.strip(), raw_text.split('|')))))

def extract_entities_from_question_file(qa_file: str, save_log_tmpdata_dir: str, meta_log: dict):
    print(qa_file)
    data = load_json(f"{EVAL_DATADIR}/{qa_file}")

    question_entities = []
    for qa_pair in tqdm(data):
        prompt = meta_log['EXTRACT_PROMPT_TEMPLATE'].format(text=qa_pair['question'])
        entities = postproc_entity_extraction(generate(prompt))
        #print(entities)
        question_entities.append({'question_entities': entities})
    
    amout_entities_per_q = list(map(lambda v: len(v['question_entities']), question_entities))
    meta_log['ENTITIES_PER_QUESTION'][qa_file] =  {
        'mean': np.mean(amout_entities_per_q), 'median': np.median(amout_entities_per_q), 'std': np.std(amout_entities_per_q),
        'min': min(amout_entities_per_q), 'max': max(amout_entities_per_q),}
    save_json(question_entities, f"{save_log_tmpdata_dir}/{qa_file}")

In [6]:
save_log_dir = f"{SAVE_STAGE1_LOGDIR}/{S1_META['SAVE_ENTITIES_VERSION']}" 
save_log_tmpdata_dir = f"{save_log_dir}/{LOG_TMPDATA_DIRNAME}"
save_log_metafile = f"{save_log_dir}/{LOG_META_FILENMAE}"

In [7]:
check_create_dir(save_log_dir)
check_create_dir(save_log_tmpdata_dir)

In [8]:
qa_files = os.listdir(EVAL_DATADIR)
for qa_file in qa_files:
    extract_entities_from_question_file(qa_file, save_log_tmpdata_dir, S1_META)

save_json(S1_META, save_log_metafile)

which_people_about_device.json


100%|██████████| 400/400 [12:18<00:00,  1.85s/it]


compare_questions.json


100%|██████████| 200/200 [05:38<00:00,  1.69s/it]


same_manufacturer.json


...

100%|██████████| 80/80 [02:09<00:00,  1.62s/it]


similar_manf_opinions.json


100%|██████████| 12/12 [00:19<00:00,  1.64s/it]


last_opinion.json


 45%|████▌     | 1430/3157 [38:34<45:48,  1.59s/it]  

#### Stage 2: connecting entities with nodes

In [5]:
S2_META = {
    "LOAD_ENTITIES_VERSION": "v1",
    "LOAD_NODES_DB_VERSION": "v3",
    "SAVE_CONN_NODES_VERSION": "v3",
    "THRESHOLD": 1.0,
    "MAX_K": 1,
    'NODES_PER_QUESTION': {}
}

NODES_DB_PATH = f'../../data/vectorized_nodes/{S2_META["LOAD_NODES_DB_VERSION"]}/densedb'
NODES_DB_META = load_json(f'../../data/vectorized_nodes/{S2_META["LOAD_NODES_DB_VERSION"]}/operation_info.json')

In [6]:
client = chromadb.PersistentClient(path=NODES_DB_PATH)
nodes_collection = client.get_collection(name=NODES_DB_META["data_name"])

embedder = HuggingFaceEmbeddings(
    model_name=NODES_DB_META["model_name"],
    model_kwargs={'device': 'cuda'},
    encode_kwargs=NODES_DB_META["encode_kwargs"]
    )

No sentence-transformers model found with name /home/dzigen/Desktop/PersonalAI/mikhail_workspace/models/facebook/contriever. Creating a new one with MEAN pooling.


In [7]:
save_log_dir = f"{SAVE_STAGE2_LOGDIR}/{S2_META['SAVE_CONN_NODES_VERSION']}" 
save_log_tmpdata_dir = f"{save_log_dir}/{LOG_TMPDATA_DIRNAME}"
save_log_metafile = f"{save_log_dir}/{LOG_META_FILENMAE}"
load_log_tmpdata_dir = f"{SAVE_STAGE1_LOGDIR}/{S2_META['LOAD_ENTITIES_VERSION']}/{LOG_TMPDATA_DIRNAME}"

qa_files = os.listdir(EVAL_DATADIR)

check_create_dir(save_log_dir)
check_create_dir(save_log_tmpdata_dir)

for qa_file in qa_files:
    print(qa_file)
    entities_data = load_json(f"{load_log_tmpdata_dir}/{qa_file}")
    
    connected_nodes = []
    for entity_item in tqdm(entities_data):
        tmp_nodes = []
        for entity in entity_item['question_entities']:
            query_embedding = embedder.embed_query(entity)

            docs_with_scores = nodes_collection.query(
                query_embeddings=[query_embedding],
                include=["documents", "metadatas", "distances"],
                n_results=50)
            
            filtered_docs_id = list(filter(lambda i: docs_with_scores['distances'][0][i] < S2_META['THRESHOLD'], 
                                           range(len(docs_with_scores['documents'][0]))))
            
            retrieved_nodes = list(map(lambda i: (
                docs_with_scores['distances'][0][i], docs_with_scores['documents'][0][i], 
                docs_with_scores['metadatas'][0][i]['node_id']), filtered_docs_id)) 
            
            if S2_META['MAX_K'] > 0:
                retrieved_nodes = retrieved_nodes[:S2_META['MAX_K']]

            tmp_nodes += retrieved_nodes
        
        unique_nodes = list(set(tmp_nodes))
        connected_nodes.append(unique_nodes)

    save_json(connected_nodes, f"{save_log_tmpdata_dir}/{qa_file}")

    amout_nodes_per_q = list(map(lambda v: len(v), connected_nodes))
    S2_META['NODES_PER_QUESTION'][qa_file] =  {
        'mean': np.mean(amout_nodes_per_q), 'median': np.median(amout_nodes_per_q), 'std': np.std(amout_nodes_per_q),
        'min': min(amout_nodes_per_q), 'max': max(amout_nodes_per_q)}

save_json(S2_META, save_log_metafile)

which_people_about_device.json


100%|██████████| 400/400 [00:23<00:00, 17.37it/s]


compare_questions.json


100%|██████████| 200/200 [00:09<00:00, 22.22it/s]


same_manufacturer.json


...

100%|██████████| 200/200 [00:06<00:00, 29.01it/s]


dominant_opinion.json


100%|██████████| 902/902 [00:42<00:00, 21.11it/s]


device_sentiment.json


100%|██████████| 200/200 [00:11<00:00, 17.23it/s]


#### Stage 3: extracting relevant triplets from knowledge graph

In [3]:
S3_META = {
    "LOAD_CONN_NODES_VERSION": "v3",
    "SAVE_TRIPLETS_VERSION": "v3",
    "LOAD_NODES_DB_VERSION": "v3",
    "MAX_DEPTH": 30,
    "MAX_WIDTH": -1,
    "SAMPLE_MAX_SIZE": 20,
    'TRIPLETES_PER_QUESTION': {}
}

NODES_DB_PATH = f'../../data/vectorized_nodes/{S3_META["LOAD_NODES_DB_VERSION"]}/densedb'
NODES_DB_META = load_json(f'../../data/vectorized_nodes/{S3_META["LOAD_NODES_DB_VERSION"]}/operation_info.json')

def euclidean_distance(node1, node2):
    v1 = nodes_collection.get(ids=[node1['id']], include=["embeddings"])['embeddings'][0]
    v2 = nodes_collection.get(ids=[node2['id']], include=["embeddings"])['embeddings'][0]
    
    return distance.euclidean(v1, v2)

def ip_distance(node1, node2):
    v1 = nodes_collection.get(ids=[node1['id']], include=["embeddings"])['embeddings'][0]
    v2 = nodes_collection.get(ids=[node2['id']], include=["embeddings"])['embeddings'][0]
    return np.dot(v1, v2)

distance_METRICS = {
    'l2': euclidean_distance,
    'ip': ip_distance
}

D_METRIC = distance_METRICS[NODES_DB_META['chroma_kwargs']['hnsw:space']]
H_METRIC = D_METRIC

In [4]:
client = chromadb.PersistentClient(path=NODES_DB_PATH)
nodes_collection = client.get_collection(name=NODES_DB_META["data_name"])

graph_model = Neo4jConnection(uri=NEO4J_URL, user=NEO4J_USER, pwd=NEO4J_PWD)

In [6]:
def get_nodes_tripletes(nodes_path: List[Dict], graph_model) -> List[Tuple[Dict, Dict, Dict]]:
    tripletes = []

    if len(nodes_path) > 1:
        for i in range(len(nodes_path)-1):
            node1, node2 = nodes_path[i], nodes_path[i+1]
            relations = graph_model.execute_query(f'MATCH (a)-[r]-(b) WHERE elementId(a) = "{node1["id"]}" AND elementId(b) = "{node2["id"]}"  RETURN a, r, b', db=NEO4J_DBNAME)
                
            for relation in relations:
                formated_node1 = {'name': relation['a']['name'], 'labels': list(relation['a'].labels), 'element_id': relation['a'].element_id}
                formated_node2 = {'name': relation['b']['name'], 'labels': list(relation['b'].labels), 'element_id': relation['b'].element_id}

                formated_relation = {'type': relation['r'].type, 'element_id': relation['r'].element_id}
                formated_relation.update(relation['r'])

                tripletes.append((formated_node1, formated_relation, formated_node2))
                
    elif len(nodes_path) == 1:
        node1 = nodes_path[0]
        relations = graph_model.execute_query(f'MATCH (a)-[r]-(b) WHERE elementId(a) = "{node1["id"]}" RETURN a, r, b', db=NEO4J_DBNAME)

        for relation in relations:
            formated_node1 = {'name': relation['a']['name'], 'labels': list(relation['a'].labels), 'element_id': relation['a'].element_id}
            formated_node2 = {'name': relation['b']['name'], 'labels': list(relation['b'].labels), 'element_id': relation['b'].element_id}

            formated_relation = {'type': relation['r'].type, 'element_id': relation['r'].element_id}
            formated_relation.update(relation['r'])

            tripletes.append((formated_node1, formated_relation, formated_node2))

    return tripletes

def update_tripletes_pool(new_tripletes: List[Tuple[Dict, Dict, Dict]], tripletes_pool: Dict[Tuple[str, str, str], Dict]) -> None:
    new_tripletes_ids = list(map(lambda triplete: (triplete[0]['element_id'], triplete[1]['element_id'], triplete[2]['element_id']), new_tripletes))

    for i, new_id in enumerate(new_tripletes_ids):
        if new_id not in tripletes_pool.keys():
            tripletes_pool[new_id] = new_tripletes[i]

In [7]:
def get_min_f_node(Q, f):
    node_ids = list(Q.keys())
    min_node_id = node_ids[0]
    min_f = f[min_node_id]

    for idx in range(1, len(node_ids)):
        if f[node_ids[idx]] < min_f:
            min_node_id = node_ids[idx]
            min_f = f[min_node_id]
            
    return Q[min_node_id]

def get_nodes_path(parent, U, end_node):
    if end_node['id'] not in parent.keys():
        return []
    
    path, end_flag, cur_n = [end_node], False, end_node['id']
    while not end_flag:
        next_n = parent[cur_n]
        if next_n is None:
            end_flag = True
        else:
            path.append(U[next_n])
            cur_n = next_n

    return path

def get_adjecent_nodes(base_node, graph_model):
    #print(node)
    raw_nodes = graph_model.execute_query(f'MATCH (a)-[r]-(b) WHERE elementId(a) = "{base_node["id"]}" RETURN b', db=NEO4J_DBNAME)
    formated_nodes = list({node['b'].element_id: {'id': node['b'].element_id, 'name': node['b']['name']} for node in raw_nodes}.values())
    
    #print(base_node, len(raw_nodes), len(formated_nodes ))

    return formated_nodes

def filter_adjenced_nodes(base_node, adj_nodes, distance_metric, max_width: int = 10):
    sorted_adj_n_distances = sorted(list(map(lambda node_item: (node_item[0], distance_metric(base_node, node_item[1])), enumerate(adj_nodes))), 
                                    key=lambda item: item[1])
    filtered_nodes = [adj_nodes[item[0]] for item in sorted_adj_n_distances[:max_width]]

    return filtered_nodes

def A_star_search(start_node, end_node, distance_metric, H_METRIC, graph_model, max_depth: int=5):
    U, Q, D = {}, {start_node['id']: start_node},  {start_node['id']: 0}
    g = {start_node['id']: 0}
    f = {start_node['id']: g[start_node['id']] + H_METRIC(start_node, end_node)}
    parent = {start_node['id']: None}

    while len(Q) != 0:
        #print(len(Q))

        current = get_min_f_node(Q, f)
        if current['id'] == end_node['id']:
            break

        del Q[current['id']]
        if D[current['id']] >= max_depth:
            continue
        U[current['id']] = current

        adj_nodes = get_adjecent_nodes(current, graph_model)
        if S3_META['MAX_WIDTH'] > 0:
            adj_nodes = filter_adjenced_nodes(current, adj_nodes, distance_metric, S3_META['MAX_WIDTH'])

        break_flag = False
        for v in adj_nodes:
            #print(current, v, end_node)
            tentativeScore = g[current['id']] + distance_metric(current, v)      
            if v['id'] in U.keys() and tentativeScore >= g[v['id']]:
                continue
            if v['id'] not in U.keys() or tentativeScore < g[v['id']]:
                parent[v['id']] = current['id']
                g[v['id']] = tentativeScore
                f[v['id']] = g[v['id']] + H_METRIC(v, end_node)
                D[v['id']] = D[current['id']] + 1

                if v['id'] not in Q.keys():
                    Q[v['id']] = v

                if D[v['id']] < max_depth and v['id'] == end_node['id']:
                    U[v['id']] = v
                    del Q[v['id']]
                    break_flag = True
                    #print(D[v['id']])
                    break

        if break_flag:
            break

    return U, Q, D, parent

In [ ]:
def extrac_relevant_tripletes_from_graph(qa_file: str, load_log_tmpdata_dir: str, save_log_tmpdata_dir: str):
    print(qa_file)

    nodes_data = load_json(f"{load_log_tmpdata_dir}/{qa_file}")

    retrieved_tripletes = []
    for item in tqdm(nodes_data[:S3_META["SAMPLE_MAX_SIZE"]]):
        formated_nodes = [{'id': node[2], 'name': node[1]} for node in item]

        tripletes_pool = {}
        if len(formated_nodes) > 1:
            for i in range(len(formated_nodes)-1):
                start_node = formated_nodes[i]
                for j in range(i+1, len(formated_nodes)):
                    end_node = formated_nodes[j]

                    U, Q, D, parent = A_star_search(start_node, end_node, D_METRIC, H_METRIC, graph_model, 
                                                    max_depth=S3_META['MAX_DEPTH'])
                    nodes_path = get_nodes_path(parent, U, end_node)
                    new_tripletes = get_nodes_tripletes(nodes_path, graph_model)

                    update_tripletes_pool(new_tripletes, tripletes_pool)

        elif len(formated_nodes) == 1:
            new_tripletes = get_nodes_tripletes(formated_nodes, graph_model)
            update_tripletes_pool(new_tripletes, tripletes_pool)
                
        retrieved_tripletes.append({'retrieved_tripletes': list(tripletes_pool.values())})
        
    amout_tripletes_per_q = list(map(lambda v: len(v['retrieved_tripletes']), retrieved_tripletes))
    S3_META['TRIPLETES_PER_QUESTION'][qa_file] =  {
        'mean': np.mean(amout_tripletes_per_q), 'median': np.median(amout_tripletes_per_q), 
        'std': np.std(amout_tripletes_per_q), 'min': np.min(amout_tripletes_per_q),
        'max': np.max(amout_tripletes_per_q)}

    save_json(retrieved_tripletes, f"{save_log_tmpdata_dir}/{qa_file}")

In [9]:
save_log_dir = f"{SAVE_STAGE3_LOGDIR}/{S3_META['SAVE_TRIPLETS_VERSION']}" 
save_log_tmpdata_dir = f"{save_log_dir}/{LOG_TMPDATA_DIRNAME}"
save_log_metafile = f"{save_log_dir}/{LOG_META_FILENMAE}"
load_log_tmpdata_dir = f"{SAVE_STAGE2_LOGDIR}/{S3_META['LOAD_CONN_NODES_VERSION']}/{LOG_TMPDATA_DIRNAME}"

In [ ]:
check_create_dir(save_log_dir)
check_create_dir(save_log_tmpdata_dir)

In [ ]:
qa_files = os.listdir(EVAL_DATADIR)
for qa_file in qa_files:
    extrac_relevant_tripletes_from_graph(qa_file, load_log_tmpdata_dir, save_log_tmpdata_dir)

save_json(S3_META, save_log_metafile)

#### Stage 4: generating answers for a questions 

In [116]:
prompt_qa_template = """You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
{c}


Question: {q}
Answer: """

In [129]:
S4_META = {
    "LOAD_TRIPLETS_VERSION": "v2",
    "SAVE_GEN_ANSWERS_VERSION": "v2",
    "ANSWER_PROMPT_TEMPLATE": prompt_qa_template,
    "TRPLETE_PROMPT_TEMPLATES": {
        "manufacturer": "- Manufacturer of {d} device is {c} company",
        "opinion": "- {p} has {s} opinion with explanation '{o}' on {t} about {f} feature of {d} device.",
        'has_device': "- {p} has {d} device."
    },
    "MAX_LIST_LEN": 100
}

sentimnets_map = {
    'pos': 'positive',
    'neg': 'negative',
    'neu': 'neutral'
}

In [130]:
def preproc_tripletes(raw_triplets: List[List[str]]) -> str:
    formated_triplets = []

    if len(raw_triplets) == 0:
        return "empty."

    for triplete in raw_triplets:
        #print(len(triplete))

        if triplete[1]["type"] == "manufacturer":
            device_node, company_node = (triplete[0], triplete[2]) if 'device' in triplete[0]["labels"] else (triplete[2], triplete[0])
            formated_triplete = S4_META["TRPLETE_PROMPT_TEMPLATES"]["manufacturer"].format(d=device_node['name'], c=company_node['name'])

        elif triplete[1]["type"] == "opinion":
            device_node, feature_node = (triplete[0], triplete[2]) if 'device' in triplete[0]["labels"] else (triplete[2], triplete[0])
            relation = triplete[1]
            formated_triplete = S4_META["TRPLETE_PROMPT_TEMPLATES"]["opinion"].format(
                p=relation["person"], o=relation["opinion"].replace('_', ' '), s=sentimnets_map.get(relation["sentiment"], relation["sentiment"]), t=relation["time"],
                f=feature_node['name'],d=device_node['name'])
        
        elif triplete[1]["type"] == "has_device":
            device_node, person_node = (triplete[0], triplete[2]) if 'device' in triplete[0]["labels"] else (triplete[2], triplete[0])
            formated_triplete = S4_META["TRPLETE_PROMPT_TEMPLATES"]['has_device'].format(d=device_node['name'], p=person_node['name'])

        else:
            raise ValueError
        
        formated_triplets.append(formated_triplete)

    return "\n".join(formated_triplets[:S4_META["MAX_LIST_LEN"]])

def generate_answers_from_triplets_file(qa_file: str, load_log_tmpdata_dir: str, save_log_tmpdata_dir: str):
    triplets_data = load_json(f"{load_log_tmpdata_dir}/{qa_file}")
    questions_data = load_json(f"{EVAL_DATADIR}/{qa_file}")

    generated_answers = []
    for triplete_item, quiestion_item in tqdm(zip(triplets_data, questions_data)):
        context = preproc_tripletes(triplete_item['retrieved_tripletes'])
        question = quiestion_item['question']
        prompt = S4_META["ANSWER_PROMPT_TEMPLATE"].format(q=question, c=context)
        
        print()
        print(prompt)
        
        answer = generate(prompt).strip()

        print(answer)

        generated_answers.append({'generated_answer': answer})
        
    save_json(generated_answers, f"{save_log_tmpdata_dir}/{qa_file}")

In [ ]:
save_log_dir = f"{SAVE_STAGE4_LOGDIR}/{S4_META['SAVE_GEN_ANSWERS_VERSION']}" 
save_log_tmpdata_dir = f"{save_log_dir}/{LOG_TMPDATA_DIRNAME}"
save_log_metafile = f"{save_log_dir}/{LOG_META_FILENMAE}"
load_log_tmpdata_dir = f"{SAVE_STAGE3_LOGDIR}/{S4_META['LOAD_TRIPLETS_VERSION']}/{LOG_TMPDATA_DIRNAME}"

In [ ]:
check_create_dir(save_log_dir)
check_create_dir(save_log_tmpdata_dir)

In [131]:
qa_files = os.listdir(EVAL_DATADIR)
for qa_file in tqdm(qa_files):
    generate_answers_from_triplets_file(
        qa_file, load_log_tmpdata_dir, save_log_tmpdata_dir)
    
save_json(S4_META, save_log_metafile)

  0%|          | 0/10 [00:00<?, ?it/s]


You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Belinda has positive opinion with explanation 'very good' on 25.9.2019 about wireless_charging feature of 10pro device.
- Brandon has negative opinion with explanation 'Slow and unreliable' on 19.10.2019 about wireless_charging feature of Xiaomi device.
- Brandon has neutral opinion with explanation 'Not a dealbreaker' on 6.2.2020 about wireless_charging feature of Xiaomi device.
- Clifford has neutral opinion with explanation 'Its okay' on 30.11.2020 about video feature of Xiaomi device.
- Alexander has neutral opinion with explanation 'Its decent' on 6.12.2020 about video feature of 

b'{"response":"None."}'
None.

You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Manufacturer of Xiaomi device is Xiaomi company
- Clifford has neutral opinion with explanation 'Its okay' on 30.11.2020 about video feature of Xiaomi device.
- Alexander has neutral opinion with explanation 'Its decent' on 6.12.2020 about video feature of Xiaomi device.
- Alexander has negative opinion with explanation 'Always shaky' on 11.11.2020 about video feature of Xiaomi device.


Question: Which people have positive opinion about video of Xiaomi 10Pro on 25.11.2020?
Answer: 


b'{"response":"None."}'
None.

You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Jenna has negative opinion with explanation 'better' on 16.10.2020 about screen feature of 10ultra device.
- Francis has neutral opinion with explanation 'Too big sometimes' on 18.11.2020 about screen feature of s21u device.
- Jennifer has positive opinion with explanation 'so vibrant' on 2.12.2020 about screen feature of v20 device.
- Jake has positive opinion with explanation 'Screen is amazing' on 31.12.2020 about screen feature of Xiaomi_Mi_12 device.
- Lynn has negative opinion with explanation 'Not clear' on 10.7.2020 about screen feature of OnePlus 

b'{"response":"None."}'
None.

You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Alyssa has negative opinion with explanation 'crazy' on 17.4.2020 about signal feature of iPhone device.
- Jessica has positive opinion with explanation 'Always been strong' on 25.11.2020 about signal feature of Apple device.
- Jennifer has positive opinion with explanation 'Really good signal' on 8.12.2020 about signal feature of Xiaomi device.
- Gerld has negative opinion with explanation 'Always have issues' on 19.6.2020 about signal feature of iPhone device.
- Kayla has positive opinion with explanation 'No signal problem' on 27.12.2020 about signal fe

...

b'{"response":"Neutral"}'
Neutral

You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Simon has 13promax device.
- Luccile has positive opinion with explanation 'Lasts all day' on 10.7.2020 about battery feature of 13promax device.
- Luccile has positive opinion with explanation 'Lasts all day' on 22.6.2020 about battery feature of 13promax device.
- Simon has positive opinion with explanation 'Its battery incredible' on 3.9.2020 about battery feature of 13promax device.
- William has negative opinion with explanation 'Really disappointing' on 14.12.2020 about battery feature of Samsung device.
- Aidan has positive opinion with explanat

b'{"response":"Negative"}'
Negative

You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Jacob has MIX4 device.


Question: Jacob has positive, negative or neutral opinion about photography camera module of MIX4 on 11.4.2020?
Answer: 


b'{"response":"Neutral"}'
Neutral

You are an expert system that can answer a question based on a given list of related information below. Answer the question in a straight, direct and short format. Do not add discriptive information about the answer. Do not add references to the given list of relatated information in your answer. Do not add additional information that is not related to the given question.

List of related information:
- Xavier has 10pro device.


Question: Xavier has positive, negative or neutral opinion about whole back panel of 10pro on 30.7.2020?
Answer: 


20it [00:28,  1.44s/it]
100%|██████████| 10/10 [04:47<00:00, 28.80s/it]

b'{"response":"Neutral"}'
Neutral


#### Stage Final: Measuring quality of generated answers

In [132]:
SF_META = {
    'SAVE_SCORES_VERSION': "v2",
    'LOAD_GEN_ANSWERS_VERSION': "v2",
    'BERT_SCORE_MODEL_PATH': "google/electra-base-discriminator"
}

BASE_DIR = '/home/dzigen/Desktop/PersonalAI/mikhail_workspace'
METRICS = ReaderMetrics(base_dir=BASE_DIR, bs_model_path=SF_META['BERT_SCORE_MODEL_PATH'])

save_log_dir = f"{SAVE_SCORES_LOGDIR}/{SF_META['SAVE_SCORES_VERSION']}" 
save_log_metafile = f"{save_log_dir}/{LOG_META_FILENMAE}"
save_log_tmpdata_dir = f"{save_log_dir}/{LOG_SCORES_DIRNAME}"
gen_answers_dir = f'{SAVE_STAGE4_LOGDIR}/{SF_META["LOAD_GEN_ANSWERS_VERSION"]}/{LOG_TMPDATA_DIRNAME}'

In [ ]:
def measure_quality_from_answers_file(qa_file: str, gen_answers_dir: str, save_log_tmpdata_dir: str):
    generated_answers_data = load_json(f"{gen_answers_dir}/{qa_file}")
    target_answers_data = load_json(f"{EVAL_DATADIR}/{qa_file}")
    
    gen_answers = list(map(lambda item: item['generated_answer'], generated_answers_data)) 
    trgt_answers = list(map(lambda item: item['answer'], target_answers_data))[:len(gen_answers)]

    b1_scores = METRICS.bleu1(gen_answers, trgt_answers)
    b2_scores  = METRICS.bleu2(gen_answers, trgt_answers)
    rl_scores = METRICS.rougel(gen_answers, trgt_answers)
    m_scores = METRICS.meteor(gen_answers, trgt_answers)
    em_scores = METRICS.exact_match(gen_answers, trgt_answers)
    bs_scores = METRICS.bertscore(gen_answers, trgt_answers)

    scores = {
        'BLEU1': str(round5(np.mean(b1_scores))),
        'BLEU2': str(round5(np.mean(b2_scores))),
        'METEOR': str(round5(np.mean(m_scores))),
        'RougeL': str(round5(np.mean(rl_scores))),
        'ExactMatch': str(round5(np.mean(em_scores))),
        'BertScore': {k: str(round5(float(v.mean()))) for k, v in bs_scores.items() if k != 'hash'}
    }

    torch.cuda.empty_cache()
    gc.collect()

    save_json(scores, f"{save_log_tmpdata_dir}/{qa_file}")

In [ ]:
check_create_dir(save_log_dir)
check_create_dir(save_log_tmpdata_dir)

In [134]:
qa_files = os.listdir(EVAL_DATADIR)
for qa_file in tqdm(qa_files):
    save_log_tmpdata_dir(qa_file, gen_answers_dir, save_log_tmpdata_dir)

save_json(SF_META, save_log_metafile)

100%|██████████| 10/10 [00:19<00:00,  1.96s/it]


#### In-place measure